# 03 — Simulación del ABM

Corre el ABM de cuencas bajo distintos escenarios ENSO y valida contra SIMMA.

**Secciones:**
1. Construir modelo desde disco
2. Escenario histórico (2010-2012) + validación vs SIMMA
3. Comparación La Niña vs El Niño vs Neutro
4. Monte Carlo con ruido estocástico
5. Mapa de activaciones por cuenca

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from abm_enso.data import cuencas as cuencas_mod
from abm_enso.model import ModeloCuencas, escenarios, validacion
from abm_enso.utils.paths import FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Cargar cuencas

In [ ]:
gdf = cuencas_mod.load()
print(f'{len(gdf)} cuencas cargadas')
if 'area_hidrografica' in gdf.columns:
    print(gdf['area_hidrografica'].value_counts())

## 2. Escenario histórico 2010-2012 + validación SIMMA

In [ ]:
oni_hist = escenarios.escenario_historico('2010-01-01', '2012-12-31')
m_hist = ModeloCuencas(gdf, oni_hist, seed=42, ruido_precip=0.15)
m_hist.run()
df_hist = m_hist.resumen_temporal()
df_hist.head()

In [ ]:
resultado = validacion.validar_contra_simma(m_hist, '2010-01', '2012-12')
print(f'r    = {resultado["r"]:.3f}')
print(f'RMSE = {resultado["rmse"]:.2f}')
print(f'F1   = {resultado["f1"]:.3f}')
print(f'n meses = {resultado["n_meses"]}')

In [ ]:
# Plot: ONI, activaciones simuladas, eventos SIMMA observados
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('ONI (forzamiento)',
                                    'Cuencas activas / mes (simulado)',
                                    'Eventos SIMMA / mes (observado)'))

fig.add_trace(go.Scatter(x=df_hist['fecha'], y=df_hist['oni'], line=dict(color='#1A3A6B'), name='ONI'), row=1, col=1)
fig.add_hline(y=0, line=dict(color='gray', width=0.5), row=1, col=1)

fig.add_trace(go.Scatter(x=df_hist['fecha'], y=df_hist['n_activaciones'],
                          line=dict(color='#0F766E'), name='Sim'), row=2, col=1)

fig.add_trace(go.Scatter(x=resultado['serie_obs'].index, y=resultado['serie_obs'].values,
                          line=dict(color='#C05050'), name='SIMMA'), row=3, col=1)

fig.update_layout(height=700, width=1000, showlegend=False,
                  title=f'Validación histórica · r={resultado["r"]:.2f}, F1={resultado["f1"]:.2f}')
fig.show()

## 3. Comparación La Niña vs El Niño vs Neutro

In [ ]:
escs = {
    'La Niña 2010': escenarios.escenario_nina_2010(36),
    'El Niño 2015': escenarios.escenario_nino_2015(36),
    'Neutro': escenarios.escenario_neutro(36),
}
resultados = {}
for nombre, oni in escs.items():
    m = ModeloCuencas(gdf, oni, seed=42, ruido_precip=0.0)
    m.run()
    resultados[nombre] = m.resumen_temporal()

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
colors = {'La Niña 2010': '#1A3A6B', 'El Niño 2015': '#C05050', 'Neutro': '#6B7280'}
for i, (nombre, df) in enumerate(resultados.items()):
    axes[0].plot(df['tick'], df['oni'], color=colors[nombre], label=nombre, lw=1.5)
    axes[1].plot(df['tick'], df['humedad_media'], color=colors[nombre], label=nombre, lw=1.5)
    axes[2].plot(df['tick'], df['n_activaciones'], color=colors[nombre], label=nombre, lw=1.5)

axes[0].set_ylabel('ONI (°C)'); axes[0].legend(); axes[0].axhline(0, color='k', lw=0.3)
axes[1].set_ylabel('Humedad media (mm)')
axes[2].set_ylabel('Cuencas activas'); axes[2].set_xlabel('Tick (mes)')
for ax in axes:
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Respuesta del sistema a 3 escenarios ENSO', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_escenarios.png', dpi=150)
plt.show()

## 4. Monte Carlo (30 réplicas) con ruido estocástico

In [ ]:
n_replicas = 30
oni = escenarios.escenario_nina_2010(36)
corridas = []
for i in range(n_replicas):
    m = ModeloCuencas(gdf, oni, seed=i, ruido_precip=0.20)
    m.run()
    corridas.append(m.resumen_temporal()['n_activaciones'].values)

corridas = np.array(corridas)
media = corridas.mean(axis=0)
p05 = np.percentile(corridas, 5, axis=0)
p95 = np.percentile(corridas, 95, axis=0)
t = np.arange(36)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(t, p05, p95, alpha=0.3, color='#1A3A6B', label='P5-P95')
ax.plot(t, media, color='#1A3A6B', lw=2, label='Media')
for c in corridas[:10]:
    ax.plot(t, c, color='gray', alpha=0.2, lw=0.5)
ax.set_xlabel('Tick (mes)'); ax.set_ylabel('Cuencas activas')
ax.set_title(f'Monte Carlo · {n_replicas} réplicas · La Niña 2010',
             fontweight='bold', loc='left')
ax.legend(); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_monte_carlo.png', dpi=150)
plt.show()

## 5. Activaciones por cuenca

In [ ]:
# Usar la última corrida para contar activaciones por cuenca
df_act = m.activaciones_por_cuenca()
df_act.sort_values('n_eventos', ascending=False).head(15)

In [ ]:
# Agregado por área hidrográfica
if 'area_hidrografica' in df_act.columns:
    agg = df_act.groupby('area_hidrografica')['n_eventos'].agg(['sum', 'mean', 'count'])
    print(agg.sort_values('sum', ascending=False))